In [8]:
# Robust Twitter sentiment pipeline for Kaggle's twitter-entity-sentiment-analysis dataset

# 1) Install & imports
!pip install nltk --quiet

import pandas as pd
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

# 2) Helper functions
def try_load_csv(path):
    # first try default
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    # quick heuristic: if any column name looks like a long sentence, treat as wrong header
    long_header = False
    for col in df.columns:
        if isinstance(col, str) and len(col) > 40:  # header pieces longer than 40 chars => likely data row
            long_header = True
            break
    if long_header:
        df = pd.read_csv(path, header=None, dtype=str, keep_default_na=False)
    return df

def detect_text_column(df):
    """Auto-detect column with tweets/text"""
    candidates = []
    for col in df.columns:
        col_series = df[col].astype(str).replace("", pd.NA).dropna()
        if len(col_series) == 0:
            continue
        med_len = col_series.map(len).median()
        candidates.append((col, med_len))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[1], reverse=True)
    best_col, best_med = candidates[0]
    if best_med < 10:
        return None
    return best_col

def clean_tweet(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)   # remove urls
    text = re.sub(r"@\w+", " ", text)                      # remove mentions
    text = re.sub(r"#", " ", text)                         # remove hashtag symbol
    text = re.sub(r"[^a-zA-Z\s']", " ", text)              # keep letters, spaces, apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 3) Sentiment setup
sia = SentimentIntensityAnalyzer()

def get_sentiment_label(text):
    score = sia.polarity_scores(text)["compound"]
    if score > 0.05:
        return "positive"
    elif score < -0.05:
        return "negative"
    else:
        return "neutral"

# 4) Processing function
def process_file(in_path, out_path, verbose=True):
    df = try_load_csv(in_path)
    if verbose:
        print(f"Loaded '{in_path}'. Columns: {df.columns.tolist()[:10]}")
    text_col = detect_text_column(df)
    if text_col is None:
        raise ValueError(
            "❌ Could not auto-detect text column. Columns are: "
            + ", ".join([str(c) for c in df.columns])
        )
    if verbose:
        print(f"Detected text column: {text_col}")
    df["clean_text"] = df[text_col].apply(clean_tweet)
    df["sentiment"] = df["clean_text"].apply(get_sentiment_label)
    df.to_csv(out_path, index=False)
    if verbose:
        print(f"✅ Saved processed output to: {out_path}")
    return df

# 5) Correct dataset paths
train_input = "/kaggle/input/twitter-entity-sentiment-analysis/twitter_training.csv"
test_input  = "/kaggle/input/twitter-entity-sentiment-analysis/twitter_validation.csv"
train_output = "/kaggle/working/twitter_train_sentiment_output.csv"
test_output  = "/kaggle/working/twitter_test_sentiment_output.csv"

# 6) Run
df_train = process_file(train_input, train_output)
df_test  = process_file(test_input, test_output)

# 7) Summaries
print("\nTrain sentiment counts:\n", df_train["sentiment"].value_counts(dropna=False))
print("\nTest sentiment counts:\n", df_test["sentiment"].value_counts(dropna=False))

display(df_train.head(5))


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Loaded '/kaggle/input/twitter-entity-sentiment-analysis/twitter_training.csv'. Columns: [0, 1, 2, 3]
Detected text column: 3
✅ Saved processed output to: /kaggle/working/twitter_train_sentiment_output.csv
Loaded '/kaggle/input/twitter-entity-sentiment-analysis/twitter_validation.csv'. Columns: [0, 1, 2, 3]
Detected text column: 3
✅ Saved processed output to: /kaggle/working/twitter_test_sentiment_output.csv

Train sentiment counts:
 sentiment
positive    33963
negative    27419
neutral     13300
Name: count, dtype: int64

Test sentiment counts:
 sentiment
positive    496
negative    396
neutral     108
Name: count, dtype: int64


,0,1,2,3,clean_text,sentiment
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,im getting on borderlands and i will murder yo...,negative
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,i am coming to the borders and i will kill you...,negative
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,im getting on borderlands and i will kill you all,negative
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,im coming on borderlands and i will murder you...,negative
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,im getting on borderlands and i will murder yo...,negative
